In [3]:
import pandas as pd

# Load CSV
df = pd.read_csv(".\CSV\SingerToPoem.csv")

# Create grouped list per PoemId
grouped = df.groupby("PoemId")["SingerId"] \
            .apply(lambda x: ",".join(map(str, sorted(set(x))))) \
            .reset_index()

grouped = grouped.rename(columns={"SingerId": "MultipleSingers"})

# Merge back to original dataframe
df = df.merge(grouped, on="PoemId", how="left")

# Save
df.to_csv("updated_file.csv", index=False)


In [1]:
import pandas as pd

# Load CSV
df = pd.read_csv(r".\CSV\SingerToPoem.csv")

# Ensure SingerId is integer
df["SingerId"] = df["SingerId"].astype(int)

# Create numeric list per PoemId
grouped = (
    df.groupby("PoemId")["SingerId"]
      .apply(lambda x: sorted(set(x)))   # real Python list of ints
      .reset_index(name="MultipleSingers")
)

# Merge back
df = df.merge(grouped, on="PoemId", how="left")

# Save (will appear as string in CSV, unavoidable)
df.to_csv("updated_file.csv", index=False)


In [8]:
import pandas as pd

# Load CSV
df = pd.read_csv(r".\CSV\SingerToPoem.csv")

# Ensure integer
df["SingerId"] = df["SingerId"].astype(int)

# Create numeric list per PoemId
multiple = (
    df.groupby("PoemId")["SingerId"]
      .apply(lambda x: sorted(set(x)))
      .reset_index(name="MultipleSingers")
)

# Merge MultipleSingers back
df = df.merge(multiple, on="PoemId", how="left")

# Drop SingerId column
df = df.drop(columns=["SingerId"])

# Remove duplicated PoemId rows (keep first occurrence)
df = df.drop_duplicates(subset=["PoemId"])

# Save
df.to_csv("clean_poems.csv", index=False)


In [ ]:
import pandas as pd

# Load the poem table
df = pd.read_csv(r".\CSV\Diwan-Hamdan-WIP - Full_poems.csv")

# Keep only rows that have a YouTube URL
df = df[df["YouTube"].notna() & (df["YouTube"].str.strip() != "")]

# Split singers by comma and explode into individual rows
df["Singer"] = df["Singer"].astype(str).str.split(",")
df_exploded = df.explode("Singer")
df_exploded["Singer"] = df_exploded["Singer"].str.strip()

# Remove empty singer names
df_exploded = df_exploded[df_exploded["Singer"] != ""]

# Count how many poems each singer appears in (SongSum)
song_sum = (
    df_exploded.groupby("Singer")
    .size()
    .reset_index(name="SongSum")
)

# Build the final singers table
singers_df = song_sum.copy()
singers_df.insert(0, "Id", range(1, len(singers_df) + 1))
singers_df.rename(columns={"Singer": "Name"}, inplace=True)
singers_df.insert(2, "Description", "")
singers_df.insert(3, "URL", "")

# Reorder columns
singers_df = singers_df[["Id", "Name", "Description", "URL", "SongSum"]]

# Sort by SongSum descending (most prolific first) - optional, remove if you want alphabetical
singers_df = singers_df.sort_values("SongSum", ascending=False).reset_index(drop=True)
singers_df["Id"] = range(1, len(singers_df) + 1)

# Save
output_path = r".\CSV\singers_table.csv"
singers_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✓ singers_table.csv created with {len(singers_df)} singers")
print("\n--- Preview ---")
print(singers_df.head(10).to_string(index=False))

✓ singers_table.csv created with 31 singers

--- Preview ---
 Id              Name Description URL  SongSum
  1       راشد الماجد                       32
  2          ميحد حمد                       23
  3         محمد عبده                       20
  4       حسين الجسمي                       17
  5 عبدالمجيد عبدالله                       13
  6       ابوبكر سالم                        7
  7            الوسمي                        4
  8     عيضه المنهالي                        4
  9       حمد العامري                        4
 10             اريام                        4


: 